In [ ]:
!free -h

               total        used        free      shared  buff/cache   available
Mem:            12Gi       1.2Gi       8.1Gi       2.2Mi       3.6Gi        11Gi
Swap:             0B          0B          0B


In [ ]:
!git clone -b audio_flamingo_2 https://github.com/NVIDIA/audio-flamingo.git
%cd audio-flamingo/inference_HF_pretrained

Cloning into 'audio-flamingo'...
remote: Enumerating objects: 1173, done.
remote: Counting objects: 100% (151/151), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 1173 (delta 122), reused 106 (delta 104), pack-reused 1022 (from 1)
Receiving objects: 100% (1173/1173), 21.92 MiB | 18.83 MiB/s, done.
Resolving deltas: 100% (610/610), done.
/content/audio-flamingo/inference_HF_pretrained


In [ ]:
!pip install -q torch torchaudio transformers accelerate einops soundfile librosa torchlibrosa ftfy braceexpand webdataset wget einops_exts flamingo

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 108.7 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import pandas as pd

path = '/content/drive/MyDrive/cut_descriptions.parquet'
df = pd.read_parquet(path)

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 350 entries, 670 to 365
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   audio        350 non-null    object
 1   title        350 non-null    object
 2   artist       350 non-null    object
 3   description  350 non-null    object
 4   desc_len     350 non-null    int64 
 5   word_count   350 non-null    int64 
dtypes: int64(2), object(4)
memory usage: 19.1+ KB


In [ ]:
df

In [ ]:
df = df.iloc[:len(df) // 3]
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 116 entries, 670 to 39
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   audio        116 non-null    object
 1   title        116 non-null    object
 2   artist       116 non-null    object
 3   description  116 non-null    object
 4   desc_len     116 non-null    int64 
 5   word_count   116 non-null    int64 
dtypes: int64(2), object(4)
memory usage: 6.3+ KB


In [ ]:
from pathlib import Path

inference_py = Path("/content/audio-flamingo/inference_HF_pretrained/inference.py")
factory_py = Path("/content/audio-flamingo/inference_HF_pretrained/src/factory.py")
config_yaml = Path("/content/audio-flamingo/inference_HF_pretrained/configs/inference.yaml")

# ======= ПАТЧ 1: factory.py — weights_only=False для torch.load =======
factory_text = factory_py.read_text()
factory_patched = factory_text.replace(
    'torch.load(clap_config["checkpoint"], map_location = \'cpu\')',
    'torch.load(clap_config["checkpoint"], map_location=\'cpu\', weights_only=False)'
)
if factory_patched != factory_text:
    factory_py.write_text(factory_patched)
    print("✅ Патч 1 применён: weights_only=False")
else:
    print("⚠️ Патч 1: строка не найдена или уже пропатчено")

# ======= ПАТЧ 2: inference.py — HF_TOKEN =======
from google.colab import userdata
HF_TOKEN = userdata.get('HF')

inference_text = inference_py.read_text()
inference_patched = inference_text.replace(
    'YOUR_HF_TOKEN',
    HF_TOKEN
).replace('"nvidia/audio-flamingo-2"', '"nvidia/audio-flamingo-2-1.5B"')
if inference_patched != inference_text:
    inference_text = inference_patched
    inference_py.write_text(inference_text)
    print("✅ Патч 2 применён: HF_TOKEN")
else:
    print("⚠️ Патч 2: уже пропатчено или строка не найдена")

# ======= ПАТЧ 3: inference.yaml — переключение на Qwen2.5-1.5B =======
config_text = config_yaml.read_text()

config_patched = config_text.replace(
    "Qwen/Qwen2.5-3B",
    "Qwen/Qwen2.5-1.5B"
).replace(
    "Qwen/Qwen2.5-3B",
    "Qwen/Qwen2.5-1.5B"
)

if config_patched != config_text:
    config_yaml.write_text(config_patched)
    print("✅ Патч 2 применён: inference.yaml -> Qwen/Qwen2.5-1.5B")
else:
    print("⚠️ Патч 2: строка Qwen/Qwen2.5-3B не найдена или уже пропатчено")

✅ Патч 1 применён: weights_only=False
✅ Патч 2 применён: HF_TOKEN
✅ Патч 2 применён: inference.yaml -> Qwen/Qwen2.5-1.5B


In [ ]:
import numpy as np
import torch.serialization


# Регистрируем numpy-глобалы как безопасные прямо в текущей сессии
torch.serialization.add_safe_globals([
    np.core.multiarray.scalar,
    np.dtype,
    np.ndarray,
])

/tmp/ipykernel_2054/3107404843.py:7: DeprecationWarning: numpy.core is deprecated and has been renamed to numpy._core. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.multiarray.
  np.core.multiarray.scalar,


In [ ]:
'''
PROMPTS = {
    "caption_mood_style": (
        "Describe this music in one short phrase. "
        "Mention its mood and broad musical style. "
        "Output only the phrase."
    ),

    "caption_energy_rhythm": (
        "Describe this music in one short phrase. "
        "Mention its energy and rhythm or pace. "
        "Output only the phrase."
    ),

    "caption_sound": (
        "Describe the audible sound in one short phrase. "
        "Mention the most noticeable instruments, vocals, or sound texture. "
        "Output only the phrase."
    ),
}

# Copyright (c) 2025 NVIDIA CORPORATION.
#   Licensed under the MIT license.

import os
import yaml
import json
import argparse

import torch
import librosa
import numpy as np
import soundfile as sf
from pydub import AudioSegment
from safetensors.torch import load_file
from huggingface_hub import snapshot_download

from src.factory import create_model_and_transforms
from utils import Dict2Class, get_autocast, get_cast_dtype

def int16_to_float32(x):
    return (x / 32767.0).astype(np.float32)

def float32_to_int16(x):
    x = np.clip(x, a_min=-1., a_max=1.)
    return (x * 32767.).astype(np.int16)

os.environ["TOKENIZERS_PARALLELISM"] = "false"


def get_num_windows(T, sr, clap_config):

    window_length  = int(float(clap_config["window_length"]) * sr)
    window_overlap = int(float(clap_config["window_overlap"]) * sr)
    max_num_window = int(clap_config["max_num_window"])

    num_windows = 1
    if T <= window_length:
        num_windows = 1
        full_length = window_length
    elif T >= (max_num_window * window_length - (max_num_window - 1) * window_overlap):
        num_windows = max_num_window
        full_length = (max_num_window * window_length - (max_num_window - 1) * window_overlap)
    else:
        num_windows = 1 + int(np.ceil((T - window_length) / float(window_length - window_overlap)))
        full_length = num_windows * window_length - (num_windows - 1) * window_overlap

    return num_windows, full_length


def read_audio(file_path, target_sr, duration, start, clap_config):

    if file_path.endswith('.mp3'):
        audio = AudioSegment.from_file(file_path)
        if len(audio) > (start + duration) * 1000:
            audio = audio[start * 1000:(start + duration) * 1000]

        if audio.frame_rate != target_sr:
            audio = audio.set_frame_rate(target_sr)

        if audio.channels > 1:
            audio = audio.set_channels(1)

        data = np.array(audio.get_array_of_samples())
        if audio.sample_width == 2:
            data = data.astype(np.float32) / np.iinfo(np.int16).max
        elif audio.sample_width == 4:
            data = data.astype(np.float32) / np.iinfo(np.int32).max
        else:
            raise ValueError("Unsupported bit depth: {}".format(audio.sample_width))

    else:
        with sf.SoundFile(file_path) as audio:
            original_sr = audio.samplerate
            channels = audio.channels

            max_frames = int((start + duration) * original_sr)

            audio.seek(int(start * original_sr))
            frames_to_read = min(max_frames, len(audio))
            data = audio.read(frames_to_read)

            if data.max() > 1 or data.min() < -1:
                data = data / max(abs(data.max()), abs(data.min()))

        if original_sr != target_sr:
            if channels == 1:
                data = librosa.resample(data.flatten(), orig_sr=original_sr, target_sr=target_sr)
            else:
                data = librosa.resample(data.T, orig_sr=original_sr, target_sr=target_sr)[0]
        else:
            if channels != 1:
                data = data.T[0]

    if data.min() >= 0:
        data = 2 * data / abs(data.max()) - 1.0
    else:
        data = data / max(abs(data.max()), abs(data.min()))

    assert len(data.shape) == 1, data.shape
    return data

def load_audio(audio_path, clap_config):

    sr = 16000
    window_length  = int(float(clap_config["window_length"]) * sr)
    window_overlap = int(float(clap_config["window_overlap"]) * sr)
    max_num_window = int(clap_config["max_num_window"])
    duration = max_num_window * (clap_config["window_length"] - clap_config["window_overlap"]) + clap_config["window_overlap"]

    audio_data = read_audio(audio_path, sr, duration, 0.0, clap_config) # hard code audio start to 0.0
    T = len(audio_data)
    num_windows, full_length = get_num_windows(T, sr, clap_config)

    # pads to the nearest multiple of window_length
    if full_length > T:
        audio_data = np.append(audio_data, np.zeros(full_length - T))

    audio_data = audio_data.reshape(1, -1)
    audio_data_tensor = torch.from_numpy(int16_to_float32(float32_to_int16(audio_data))).float()

    audio_clips = []
    audio_embed_mask = torch.ones(num_windows)
    for i in range(num_windows):
        start = i * (window_length - window_overlap)
        audio_data_tensor_this = audio_data_tensor[:, start:start+window_length]
        audio_clips.append(audio_data_tensor_this)

    if len(audio_clips) > max_num_window:
        audio_clips = audio_clips[:max_num_window]
        audio_embed_mask = audio_embed_mask[:max_num_window]

    audio_clips = torch.cat(audio_clips)

    return audio_clips, audio_embed_mask

def predict(filepath, question, clap_config, inference_kwargs):

    audio_clips, audio_embed_mask = load_audio(filepath, clap_config)

    # ВАЖНО: не переводим вручную в bf16/fp16, оставляем float32
    audio_clips = audio_clips.to(device_id, dtype=torch.float32, non_blocking=True)
    audio_embed_mask = audio_embed_mask.to(device_id, dtype=torch.float32, non_blocking=True)

    text_prompt = str(question).lower()

    sample = f"<audio>{text_prompt.strip()}{tokenizer.sep_token}"

    text = tokenizer(
        sample,
        max_length=512,
        padding="longest",
        truncation="only_first",
        return_tensors="pt"
    )

    input_ids = text["input_ids"].to(device_id, non_blocking=True)
    attention_mask = text["attention_mask"].to(device_id, non_blocking=True)

    prompt = input_ids

    with torch.no_grad():
        output = model.generate(
            audio_x=audio_clips.unsqueeze(0),
            audio_x_mask=audio_embed_mask.unsqueeze(0),
            lang_x=prompt,
            attention_mask=attention_mask,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
            max_new_tokens=24,
            **inference_kwargs,
        )[0]

    output_decoded = tokenizer.decode(output, skip_special_tokens=False)
    output_decoded = output_decoded.split(tokenizer.sep_token)[-1]
    output_decoded = output_decoded.replace(tokenizer.eos_token, '')
    output_decoded = output_decoded.replace(tokenizer.pad_token, '')
    output_decoded = output_decoded.replace('<|endofchunk|>', '')
    output_decoded = output_decoded.strip()

    print('Prompt:', question)
    print('Audio Flamingo 2:', output_decoded)

    return output_decoded


if __name__ == "__main__":

    parser = argparse.ArgumentParser()
    parser.add_argument("--input", "-i", type=str, help="Path to input JSON file")
    parsed_args = parser.parse_args()

    snapshot_download(repo_id="nvidia/audio-flamingo-2-1.5B", local_dir="./", token="") \\\

    config = yaml.load(open("configs/inference.yaml"), Loader=yaml.FullLoader)

    data_config = config['data_config']
    model_config = config['model_config']
    clap_config = config['clap_config']
    args = Dict2Class(config['train_config'])

    # Для инференса форсируем fp32
    args.precision = "32"

    model, tokenizer = create_model_and_transforms(
        **model_config,
        clap_config=clap_config,
        use_local_files=args.offline,
        gradient_checkpointing=args.gradient_checkpointing,
        freeze_lm_embeddings=args.freeze_lm_embeddings,
    )

    device_id = 0

    print("1/5 Model architecture created", flush=True)

    with open("safe_ckpt/metadata.json", "r") as f:
        metadata = json.load(f)

    print("2/5 Loading checkpoint files into RAM", flush=True)

    state_dict = {}
    for chunk_name in metadata:
        print(f"Loading: {chunk_name}", flush=True)

        chunk_path = f"safe_ckpt/{chunk_name}.safetensors"
        chunk_tensors = load_file(chunk_path)
        state_dict.update(chunk_tensors)

    print("3/5 Applying checkpoint weights", flush=True)

    missing_keys, unexpected_keys = model.load_state_dict(
        state_dict,
        strict=False
    )

    print(
        f"Missing keys: {len(missing_keys)} | "
        f"Unexpected keys: {len(unexpected_keys)}",
        flush=True
    )

    del state_dict
    del chunk_tensors

    import gc
    gc.collect()

    print("4/5 Moving model to GPU", flush=True)

    model = model.to(device_id)
    model = model.to(torch.float32)
    model.eval()

    torch.cuda.empty_cache()

    print("5/5 Model ready", flush=True)

    '''autocast = get_autocast(
        args.precision, cache_enabled=(not args.fsdp)
    )

    cast_dtype = get_cast_dtype(args.precision)'''

    data = []
    with open(parsed_args.input, "r", encoding="utf-8") as file:
        for line in file:
            data.append(json.loads(line.strip()))


    inference_kwargs = {
    "do_sample": True,
    "temperature": 0.7,
    "top_p": 0.9,
    "top_k": 30,

    "num_return_sequences": 1,
}

    results = []

    total_jobs = len(data) * len(PROMPTS)
    job_i = 0

    for item_i, item in enumerate(data, start=1):
        for prompt_id, question in PROMPTS.items():
            job_i += 1
            print(
                f"[{job_i}/{total_jobs}] Track {item_i}/{len(data)} | "
                f"Prompt: {prompt_id} | {item['track_name']}",
                flush=True
            )

            text = predict(
                item["path"],
                question,
                clap_config,
                inference_kwargs
            )

            results.append({
                "idx": item["idx"],
                "track_name": item["track_name"],
                "prompt_id": prompt_id,
                "prompt_text": question,
                "description": text
            })

    with open("predictions_prompt_eval.jsonl", "w", encoding="utf-8") as f:
        for row in results:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    print(f"\nSaved {len(results)} predictions to predictions_prompt_eval.jsonl")

'''

In [ ]:
import tempfile
import json
import pandas as pd
from pathlib import Path

out_path = "/content/audio-flamingo/inference_HF_pretrained/inference.jsonl"

with open(out_path, "w", encoding="utf-8") as f:
    for idx, row in df.iterrows():
        audio_bytes = row["audio"]["bytes"]
        original_path = row["audio"].get("path", f"track_{idx}.mp3")
        suffix = Path(original_path).suffix or ".mp3"

        tmp = tempfile.NamedTemporaryFile(delete=False, suffix=suffix)
        tmp.write(audio_bytes)
        tmp.close()

        rec = {
            "idx": int(idx),
            "track_name": original_path,
            "path": tmp.name
        }
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

In [ ]:
%cd /content/audio-flamingo/inference_HF_pretrained/my_laion_clap/CLAP/src/laion_clap/clap_module

%ls -lh bpe_simple_vocab_16e6.txt.gz
!head -c 160 bpe_simple_vocab_16e6.txt.gz

/content/audio-flamingo/inference_HF_pretrained/my_laion_clap/CLAP/src/laion_clap/clap_module
-rw-r--r-- 1 root root 132 Sep 12 22:14 bpe_simple_vocab_16e6.txt.gz
version https://git-lfs.github.com/spec/v1
oid sha256:924691ac288e54409236115652ad4aa250f48203de50a9e4722a6ecd48d6804a
size 1356917


In [ ]:
%cd /content/audio-flamingo/inference_HF_pretrained/my_laion_clap/CLAP/src/laion_clap/clap_module

%rm -f bpe_simple_vocab_16e6.txt.gz

!wget -q --show-progress \
  -O bpe_simple_vocab_16e6.txt.gz \
  https://openaipublic.azureedge.net/clip/bpe_simple_vocab_16e6.txt.gz

%ls -lh bpe_simple_vocab_16e6.txt.gz
!file bpe_simple_vocab_16e6.txt.gz

!python - <<'PY'
import gzip

path = "bpe_simple_vocab_16e6.txt.gz"

with open(path, "rb") as f:
    magic = f.read(2)

print("First bytes:", magic)
assert magic == b"\x1f\x8b", f"ERROR: not gzip, got {magic!r}"

with gzip.open(path, "rt", encoding="utf-8") as f:
    first_line = f.readline().strip()

print("OK: valid gzip BPE vocabulary")
print("First line:", first_line)
!PY

/content/audio-flamingo/inference_HF_pretrained/my_laion_clap/CLAP/src/laion_clap/clap_module
bpe_simple_vocab_16 100%[===================>]   1.29M  --.-KB/s    in 0.05s   
-rw-r--r-- 1 root root 1.3M Dec 30  2020 bpe_simple_vocab_16e6.txt.gz
bpe_simple_vocab_16e6.txt.gz: gzip compressed data, last modified: Fri Sep 11 06:03:50 2020, from Unix, original size modulo 2^32 3194984
/bin/bash: line 1: warning: here-document at line 1 delimited by end-of-file (wanted `PY')
First bytes: b'\x1f\x8b'
OK: valid gzip BPE vocabulary
First line: "bpe_simple_vocab_16e6.txt#version: 0.2
/bin/bash: line 1: PY: command not found


In [ ]:
!apt-get update -qq
!apt-get install -y -qq git-lfs

%cd /content/audio-flamingo/inference_HF_pretrained/my_laion_clap/CLAP

!git lfs install
!git lfs pull

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package git-lfs.
(Reading database ... 126952 files and directories currently installed.)
Preparing to unpack .../git-lfs_3.4.1-1ubuntu0.4_amd64.deb ...
Unpacking git-lfs (3.4.1-1ubuntu0.4) ...
Setting up git-lfs (3.4.1-1ubuntu0.4) ...
Processing triggers for man-db (2.12.0-4build2) ...
/content/audio-flamingo/inference_HF_pretrained/my_laion_clap/CLAP
Updated Git hooks.
Git LFS initialized.


In [ ]:
%cd /content/audio-flamingo/inference_HF_pretrained

import os


assert HF_TOKEN, "❌ HF_TOKEN пустой"
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

!python inference.py --input inference.jsonl

/content/audio-flamingo/inference_HF_pretrained
/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/content/audio-flamingo/inference_HF_pretrained/src/helpers.py:178: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)
/content/audio-flamingo/inference_HF_pretrained/src/helpers.py:203: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0% 0/13 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/522 [00:00<?, ?B/s]           

Fetching 13 files:   8% 1/13 [0

In [ ]:
jsonl_path = "/content/audio-flamingo/inference_HF_pretrained/predictions_prompt_eval.jsonl"

# Чтение JSON Lines в DataFrame
df = pd.read_json(jsonl_path, lines=True)

df.head()

,idx,track_name,prompt_id,prompt_text,description
0,670,000921.mp3,caption_mood_style,Describe this music in one short phrase. Menti...,this melancholic folk song with spoken word an...
1,670,000921.mp3,caption_energy_rhythm,Describe this music in one short phrase. Menti...,the track has a medium tempo
2,670,000921.mp3,caption_sound,Describe the audible sound in one short phrase...,"high-energy, fast-paced sound with prominent v..."
3,594,000840.mp3,caption_mood_style,Describe this music in one short phrase. Menti...,"this upbeat pop track features a catchy, dance..."
4,594,000840.mp3,caption_energy_rhythm,Describe this music in one short phrase. Menti...,this energetic track is perfect for dancing wi...


In [ ]:
output_path = "/content/drive/MyDrive/sintetic_intents_3.csv"

df.to_csv(output_path, index=False)

print("Сохранено:", output_path)

Сохранено: /content/drive/MyDrive/sintetic_intents_3.csv
